In [1]:
# ✅ Step 1: System & Directory Setup
import os

# Define base path
base_path = "/content/surveillance_project"

# Create directories
os.makedirs(base_path, exist_ok=True)
os.makedirs(os.path.join(base_path, "checkpoints"), exist_ok=True)

print("📁 Project directories created at:", base_path)


📁 Project directories created at: /content/surveillance_project


In [ ]:
# 📦 STEP 1: Clean the environment (run only once per session)
!pip uninstall -y torch torchvision torchaudio jax jaxlib tensorflow numpy gymnasium thinc blosc2 pymc dopamine-rl treescope xarray-einstats

# 🚀 STEP 2: Install required stable versions
!pip install "torch==2.0.1" "numpy==1.24.4" "gymnasium==0.29.1" "stable-baselines3[extra]"

# 🔁 STEP 3: Restart runtime (very important)
import os
os.kill(os.getpid(), 9)


Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: jax 0.5.2
Uninstalling jax-0.5.2:
  Successfully uninstalled jax-0.5.2
Found existing installation: jaxlib 0.5.1
Uninstalling jaxlib-0.5.1:
  Successfully uninstalled jaxlib-0.5.1
Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: gymnasium 1.1.1
Uninstalling gymnasium-1.1.1:
  Successfully uninstalled gymnasium-1.1.1
Found existing installation: thinc 

In [1]:
# ✅ Step 3: Validate Installation
import torch
import gymnasium as gym
import stable_baselines3
from stable_baselines3.common.vec_env import VecNormalize

print("🚀 PyTorch version:", torch.__version__)
print("✅ Gymnasium version:", gym.__version__)
print("✅ Stable Baselines3 version:", stable_baselines3.__version__)
print("🔥 CUDA Available:", torch.cuda.is_available())


🚀 PyTorch version: 2.0.1+cu117
✅ Gymnasium version: 0.29.1
✅ Stable Baselines3 version: 2.4.1
🔥 CUDA Available: False


In [7]:
import os
import numpy as np
import gymnasium as gym
import pandas as pd
import torch
import time
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

### --- Surveillance Drone Environment ---
class SurveillanceDroneEnv(gym.Env):
    def __init__(self, obstacle_csv="/content/surveillance_project/obstacle_positions.csv"):
        super().__init__()

        self.area_min = np.array([-3500, -11750, 0])
        self.area_max = np.array([11500, 11250, 2500])
        self.grid_res = 1000

        self.grid = self._generate_grid_points()

        self.obstacles = set()
        self.visited = set()
        self.global_visited = set()
        self.current_idx = 0
        self._load_obstacles_from_csv(obstacle_csv)

        self.observation_space = gym.spaces.Box(low=-10000, high=10000, shape=(3,), dtype=np.float32)
        self.action_space = gym.spaces.Discrete(6)
        self.visited_points = {}
        self.prev_position = None
        self.total_steps = 0

        self.max_steps = 1500
        self.step_count = 0
        self.collisions = 0
        self.revisits = 0
        self.total_possible_points = self._calculate_total_possible_points()

    def _generate_grid_points(self):
        x = np.arange(self.area_min[0], self.area_max[0], self.grid_res)
        y = np.arange(self.area_min[1], self.area_max[1], self.grid_res)
        z = np.arange(self.area_min[2], self.area_max[2], self.grid_res)
        return np.array(np.meshgrid(x, y, z)).T.reshape(-1, 3)

    def _calculate_total_possible_points(self):
        return len(self.grid)

    def _load_obstacles_from_csv(self, file_path):
        df = pd.read_csv(file_path)
        for _, row in df.iterrows():
            if str(row['Name']).lower() == 'ground':
                continue

            center = np.array([row['X'], row['Y'], row['Z']])
            dims = np.array([row['dimx'], row['dimy'], row['dimz']])

            min_corner = center - dims / 2
            max_corner = center + dims / 2

            x_vals = np.arange(min_corner[0], max_corner[0] + self.grid_res, self.grid_res)
            y_vals = np.arange(min_corner[1], max_corner[1] + self.grid_res, self.grid_res)
            z_vals = np.arange(min_corner[2], max_corner[2] + self.grid_res, self.grid_res)

            for x in x_vals:
                for y in y_vals:
                    for z in z_vals:
                        grid_pos = tuple(np.round(np.array([x, y, z]) / self.grid_res * self.grid_res).astype(int))
                        self.obstacles.add(grid_pos)

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.current_idx = 0
        self.visited.clear()
        self.episode_reward = 0
        self.revisits = 0
        self.collisions = 0
        self.visited_points.clear()
        self.prev_position = None
        self.total_steps = 0
        return self.grid[self.current_idx], {}

    def step(self, action):
        self.step_count += 1
        self.total_steps += 1

        dx = np.array([[self.grid_res, 0, 0], [-self.grid_res, 0, 0],
                       [0, self.grid_res, 0], [0, -self.grid_res, 0],
                       [0, 0, self.grid_res], [0, 0, -self.grid_res]])

        next_pos = self.grid[self.current_idx] + dx[action]
        grid_tuple = tuple(np.round(next_pos / self.grid_res * self.grid_res).astype(int))
        self.global_visited.add(tuple(self.grid[self.current_idx]))
        reward = 0
        if grid_tuple in self.obstacles or not self._is_valid(next_pos):
            self.collisions += 1
            reward = -1
        else:
            if tuple(self.grid[self.current_idx]) in self.visited:
                self.revisits += 1
            self.visited.add(tuple(self.grid[self.current_idx]))
            self.current_idx += 1
            reward = 1

        if self.current_idx >= len(self.grid) - 1:
            reward += 50

        grid_pos = tuple(np.round(self.grid[self.current_idx] / self.grid_res).astype(int))
        self.visited_points[grid_pos] = self.visited_points.get(grid_pos, 0) + 1

        if self.prev_position is not None:
            dist_moved = np.linalg.norm(self.grid[self.current_idx] - self.prev_position)
            if dist_moved < self.grid_res * 0.5:
                reward -= 0.1

        revisit_count = sum(1 for v in self.visited_points.values() if v > 1)
        unique_points = len(self.visited_points)
        revisit_ratio = revisit_count / unique_points if unique_points > 0 else 0
        reward += 0.5 * (1 - revisit_ratio)
        self.episode_reward += reward
        self.prev_position = self.grid[self.current_idx].copy()
        done = self.step_count >= self.max_steps or self.current_idx >= len(self.grid) - 1
        return self.grid[self.current_idx], reward, done, False, {}

    def _is_valid(self, pos):
        return np.all(pos >= self.area_min) and np.all(pos <= self.area_max)


### --- Logger Callback ---
class CSVLogger(BaseCallback):
    def __init__(self, log_path, verbose=0):
        super().__init__(verbose)
        self.log_path = log_path
        self.logs = []
        self.start_time = time.time()

    def _on_step(self):
      if self.n_calls % self.model.n_steps == 0:
          # Unwrap to get the actual environment
          vec_env = self.model.get_env()
          raw_env = vec_env.envs[0]  # This is the actual SurveillanceDroneEnv instance

          total_cells = len(raw_env.grid)
          total_obstacles = len(raw_env.obstacles)
          visited = len(raw_env.global_visited)
          valid_cells = max(total_cells - total_obstacles, 1)

          area_coverage = min(visited / valid_cells, 1.0)
          obstacle_avoidance = 1 - (raw_env.collisions / raw_env.step_count) if raw_env.step_count else 1
          path_efficiency = visited / raw_env.step_count if raw_env.step_count else 0
          exploration_diversity = visited / total_cells
          elapsed_time = time.time() - self.start_time
          # Cumulative
          global_visited = len(raw_env.global_visited)
          global_area_coverage = min(global_visited / valid_cells, 1.0)

          # Per-episode
          episode_visited = len(raw_env.visited)
          episode_area_coverage = min(episode_visited / valid_cells, 1.0)
          # reward tracker
          episode_rewards = [env.episode_reward for env in self.model.get_env().envs]
          avg_reward = np.mean(episode_rewards)

          explained_var = float(self.model.logger.name_to_value.get('rollout/explained_variance', 0))
          value_loss = float(np.mean(self.model.logger.name_to_value.get('train/value_loss', 0)))

          logs = {
              "timestep": self.num_timesteps,
              "area_coverage": round(area_coverage, 4),
              "obstacle_avoidance": round(obstacle_avoidance, 4),
              "path_efficiency": round(path_efficiency, 4),
              "revisits": raw_env.revisits,
              "collisions": raw_env.collisions,
              "exploration_diversity": round(exploration_diversity, 4),
              "avg_reward": round(avg_reward, 4),
              "global_area_coverage": round(global_area_coverage, 4),
              "episode_area_coverage": round(episode_area_coverage, 4),
              "explained_variance": round(explained_var, 4),
              "value_loss": round(value_loss, 4),
              "elapsed_time": round(elapsed_time, 2)
          }

          self.logs.append(logs)
      return True


    def _on_training_end(self):
        pd.DataFrame(self.logs).to_csv(self.log_path, index=False)




def train_drone():
    env = DummyVecEnv([lambda: SurveillanceDroneEnv()])
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)
    env.training = True

    # ✅ Correctly unwrapped environment
    raw_env = env.venv.envs[0]
    raw_env.global_visited.clear()

    model = PPO("MlpPolicy", env, verbose=1)
    logger = CSVLogger("/content/surveillance_project/training_log.csv")
    model.learn(total_timesteps=100_000, callback=logger)

    model.save("/content/surveillance_project/surveillance_drone_model")
    env.save("/content/surveillance_project/vec_normalize.pkl")

    print("✅ Model saved as 'surveillance_drone_model.zip'")
    print("📄 Logs saved to 'training_log.csv'")
    print("🔹 VecNormalize stats saved to 'vec_normalize.pkl'")



if __name__ == "__main__":
    train_drone()


Using cpu device
-----------------------------
| time/              |      |
|    fps             | 707  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 495         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008420469 |
|    clip_fraction        | 0.0701      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.79       |
|    explained_variance   | -0.00968    |
|    learning_rate        | 0.0003      |
|    loss                 | -0.00681    |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00866    |
|    value_loss           | 0.387       |
-----------------------------------------
-----------------

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# === LOAD VISITED POINTS ===
with open("/content/surveillance_project/visited_points.pkl", "rb") as f:
    visited = pickle.load(f)  # dict with (x, y, z) keys

# === PROJECT TO 2D (x, y) ===
x_y_points = [(x, y) for (x, y, z) in visited.keys()]

# === COUNT FREQUENCIES ===
df = pd.DataFrame(x_y_points, columns=["x", "y"])
heat = df.value_counts().reset_index(name="visits")

# === PIVOT TO 2D GRID ===
pivot_table = heat.pivot(index="y", columns="x", values="visits").fillna(0)

# === PLOT HEATMAP ===
plt.figure(figsize=(12, 10))
sns.heatmap(pivot_table, cmap="viridis", cbar=True, linewidths=0.2, linecolor='gray')
plt.title("🛰️ Drone Area Coverage Heatmap (Top-Down XY View)", fontsize=16)
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.gca().invert_yaxis()  # To match traditional map view
plt.tight_layout()
plt.savefig("/content/surveillance_project/drone_heatmap.png")
plt.show()
